# transistor：前処理照合・角度を変えた復元・ランダム回転学習の比較

**対象は全てtransistorです。bottleへの効果はまだ結論にしません。**

1. **再学習なし**：過去の2モデルで、Pillow直接処理の回転なし／既知角度入力・スコア・AUROCを保存済み結果と照合する。
2. **再学習なし**：正常画像の角度を0〜72度まで9度ずつ変え、復元の向き・細部・誤差を観察する。
3. **新規学習あり**：通常学習／固定回転／毎回ランダム回転を、同じ初期重み・6,700更新・滑らかな共通βで学習して比較する。

**コード作成・静的確認のみ。アシスタントはセル・学習・推論・テストを実行していません。**
上から実行すると、第3部で正式な3モデルの学習を開始します（初期値 `RUN_FULL_TRAINING=True`）。
まず1・2だけ確認したい場合は、第3部の学習セルより前で止めてください。
第1部が不一致なら、保存された照合表を確認し、原因を解決してから進めます。

全処理でtorchvisionはimportせず、Pillow直接処理を使います。既存のモデル・結果・元画像は上書きしません。
各回の実行は新しいセッションフォルダへ保存します。高解像度で毎回回転するCの学習は、前処理にも時間がかかります。
学習画像のRGB保持や固定回転キャッシュ、保存する入出力配列にもメモリ・ディスク容量を使用します。


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
from io import BytesIO
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlib
import importlib.metadata
import json
import math
import platform
import random
import uuid
import csv
import time
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display
from skimage.metrics import structural_similarity as ssim

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p/'memory/memory.md').is_file() and (p/'data/transistor').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('このリポジトリ内を作業ディレクトリにしてください。')
NORMAL_RUN = ROOT/'saved_models/transistor_update_matched_20261106/20261106_021112_b266ecab'
ROTATION_RUN = ROOT/'saved_models/transistor_rotation_control_20261106_reflect/20261106_002435_9b4da165'
BASELINE_DIR = NORMAL_RUN/'evaluations/20261106_025126_58aad9c8'
KNOWN_DIR = ROOT/'saved_models/transistor_rotated_test_20261106/20261106_031022_6bed84fa'
UNSEEN_RUN = ROOT/'saved_models/transistor_unseen_rotation_20261117/20261117_232200_a212593a'
OUTPUT_BASE = ROOT/'saved_models/transistor_rotation_followup_20261118'
MODEL_PATHS = {'A_original':NORMAL_RUN/'model_final.pth',
               'B_rotate10':ROTATION_RUN/'seed_42/B_rotate10/model_final.pth'}
EXPECTED_HASHES = {'A_original':'380bb28d611612ecda31f97dee299c2310e1b39f6c52b2fbd92144e2ab3e4667',
                   'B_rotate10':'20b0568e0cbae99290cb474b7e327a95983368fb37ece7edbcd0cd1c91f42de5'}
CONDITIONS = tuple(MODEL_PATHS)
LABELS = {'A_original':'A：通常学習','B_rotate10':'B：回転拡張'}
COLORS = {'A_original':'#526b8e','B_rotate10':'darkorange'}
SCENARIO_LABELS = {'unrotated':'回転なし',
    'known_rotated':'既知角度',
    'midpoint':'中間角度（5回）','unseen_random':'未知ランダム角度（5回）'}
CATEGORY_LABELS = {'good':'正常','bent_lead':'端子の曲がり','cut_lead':'端子の切断',
                   'damaged_case':'本体の損傷','misplaced':'配置ずれ'}
TRAINING_ANGLES = tuple(range(0,360,36))
ANGLE_CANDIDATES = {
    'midpoint': tuple(range(18,360,36)),
    'unseen_random': tuple(a for a in range(360) if a % 36 != 0),
}
ASSIGNMENT_SEEDS = (42,43,44,45,46)
ASSIGNMENT_SEED = 42  # 推論環境の乱数固定用。評価は平均μから行う。
REPRESENTATIVE_SEED = 42  # ROC・ヒストグラム用。結果を見て選び直さない。
VISUAL_SCENARIOS = ('midpoint','unseen_random')
VISUAL_SEEDS = (42,)  # 別の回も例：(42,43,44,45,46)
DEVICE = torch.device('cpu')  # 既存の評価と同じ環境
EVAL_BATCH_SIZE = 32
TOP_K = 130
# 以下は可視化だけの設定。評価は全条件・全5回・各100枚です。
VISUAL_PATHS = None  # 例：['good/000.png','cut_lead/000.png']。Noneなら種類ごとの先頭画像
MAX_IMAGES_PER_CATEGORY = 1  # VISUAL_PATHS=Noneのとき。Noneなら全画像

random.seed(ASSIGNMENT_SEED)
np.random.seed(ASSIGNMENT_SEED)
torch.manual_seed(ASSIGNMENT_SEED)
torch.set_num_threads(min(4,max(1,torch.get_num_threads())))
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
for font in ['Yu Gothic','Meiryo','MS Gothic','Noto Sans CJK JP','IPAexGothic']:
    if font in available_fonts:
        plt.rcParams['font.family'] = font
        break
else:
    import japanize_matplotlib
plt.rcParams.update({'font.size':11,'axes.unicode_minus':False,'figure.dpi':100})
def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))
def write_json(path,value):
    path.write_text(json.dumps(value,ensure_ascii=False,indent=2)+chr(10),encoding='utf-8')
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


# 学習済み重みから評価を再開する場合だけFalseと実行IDを指定する。
RUN_FULL_TRAINING = True
TRAINING_RUN_ID = None  # 例：'20261118_...'; OUTPUT_BASE直下のフォルダ名のみ
SEED = 42
TARGET_UPDATES = 6700
WARMUP_UPDATES = 3350
BETA_MAX = 1.0
LATENT_DIM = 64
BATCH_SIZE = 32
LEARNING_RATE = 0.001
TRAIN_CONDITIONS = ('A_normal','B_fixed','C_random')
TRAIN_LABELS = {'A_normal':'A：通常学習','B_fixed':'B：固定回転','C_random':'C：ランダム回転'}
TRAIN_COLORS = {'A_normal':'#526b8e','B_fixed':'darkorange','C_random':'#388b62'}
SCORE_ATOL = 1e-6
SCORE_RTOL = 1e-6
AUROC_ATOL = 1e-12
# 第2部だけの設定。Noneなら正常テスト全60枚。
NORMAL_SWEEP_COUNT = 5
SWEEP_ANGLES = (0,9,18,27,36,45,54,63,72)
SWEEP_COLUMNS_PER_FIG = 3

MODEL_DEFINITION_SHA = '546d83343d03107deffc7f46c4b34c4996f419f807f0a799d524a28931e4f7a2'


## 共通：モデル・保存結果・テスト画像の照合

過去のSSE版Hybridモデルとテスト100枚を使用します。定義・処理は11月17日のノートブックから継承。
初期の前処理ハッシュ照合で、回転なし入力は保存済み画像と完全一致を要求します。


In [ ]:
normal_config = read_json(NORMAL_RUN/'config.json')
rotation_config = read_json(ROTATION_RUN/'config.json')
normal_summary = read_json(NORMAL_RUN/'summary.json')
rotation_summary = read_json(MODEL_PATHS['B_rotate10'].parent/'summary.json')
assert normal_config['status'] == rotation_config['status'] == 'completed'
assert normal_summary['updates'] == rotation_summary['updates'] == 6700
assert normal_config['seed'] == normal_summary['seed'] == rotation_summary['seed'] == 42
assert normal_summary['initial_state_sha256'] == rotation_summary['initial_state_sha256']
assert normal_config['source_model_sha256'] == EXPECTED_HASHES['B_rotate10']
assert normal_config['model'] == 'Hybrid CNN + FC'
assert rotation_config['angles'] == list(TRAINING_ANGLES)
assert rotation_config['rotation_background'] == 'reflect'
assert rotation_config['rotation_interpolation'] == 'NEAREST'
assert rotation_config['rotation_expand'] is False
assert rotation_config['rotation_stage'] == 'full_resolution_before_resize'
assert rotation_config['source_size'] == [1024,1024]
assert normal_config['latent_dim'] == rotation_config['latent_dim'] == 64
assert normal_summary['model_sha256'] == EXPECTED_HASHES['A_original']
assert all(sha256(path) == EXPECTED_HASHES[name] for name,path in MODEL_PATHS.items())
source_files = [*MODEL_PATHS.values(), NORMAL_RUN/'config.json', ROTATION_RUN/'config.json',
                NORMAL_RUN/'summary.json', MODEL_PATHS['B_rotate10'].parent/'summary.json',
                ROTATION_RUN/'test_manifest.json', NORMAL_RUN/'test_manifest.json',
                BASELINE_DIR/'comparison.csv', BASELINE_DIR/'image_scores.csv']
source_hashes = {p.relative_to(ROOT).as_posix():sha256(p) for p in source_files}
baseline_comparison = pd.read_csv(BASELINE_DIR/'comparison.csv').set_index('condition')
baseline_scores = pd.read_csv(BASELINE_DIR/'image_scores.csv').set_index('relative_path')
assert np.isclose(baseline_comparison.loc['A_original','auroc'],0.82625,atol=1e-12,rtol=0)
assert np.isclose(baseline_comparison.loc['B_rotate10','auroc'],0.8066666666666666,atol=1e-12,rtol=0)


known_config = read_json(KNOWN_DIR/'config.json')
assert known_config['status'] == 'completed'
assert known_config['model_hashes'] == EXPECTED_HASHES
assert known_config['allowed_angles'] == list(range(36,360,36))
assert known_config['assignment_seed'] == 42
assert known_config['ssim_window'] == 11 and known_config['top_k'] == TOP_K
assert known_config['data_range'] == 1
known_scores = pd.read_csv(KNOWN_DIR/'image_scores.csv')
known_assignment = pd.read_csv(KNOWN_DIR/'angle_assignment.csv')
known_comparison = pd.read_csv(KNOWN_DIR/'scenario_comparison.csv')
for filename in ['config.json','image_scores.csv','angle_assignment.csv','scenario_comparison.csv','test_manifest.json']:
    path = KNOWN_DIR/filename
    source_hashes[path.relative_to(ROOT).as_posix()] = sha256(path)


In [ ]:
# 全条件で画像全体を使う。中央820の切り出しは行わない。
# torchvisionのPIL経路と同じPillow操作を直接使用する。
# torchvision全体のimport（未使用のNMS等の拡張登録）を必要としない。
def final_transform(image):
    return image.resize((128,128),resample=Image.Resampling.LANCZOS).convert('L')

def to_tensor(image):
    if image.mode != 'L':
        raise ValueError('白黒化したLモード画像を指定してください。')
    pixels = np.array(image,dtype=np.uint8,copy=True)
    return torch.from_numpy(pixels).unsqueeze(0).to(dtype=torch.float32).div(255)

def rotation_pad_width(size):
    return math.ceil((math.hypot(*size)-min(size))/2)+4

def rotate_with_reflect(image,angle,verify_support=False):
    if angle==0: return image.copy()
    width,height=image.size
    pad=rotation_pad_width(image.size)
    pixels=np.asarray(image)
    extended=Image.fromarray(np.pad(pixels,((pad,pad),(pad,pad),(0,0)),mode='reflect'))
    box=(pad,pad,pad+width,pad+height)
    options=dict(resample=Image.Resampling.NEAREST,expand=False,center=None)
    out=extended.rotate(angle,fillcolor=(255,255,255),**options).crop(box)
    mask=Image.new('L',image.size,255).rotate(angle,fillcolor=0,**options)
    original_part=image.rotate(angle,fillcolor=(255,255,255),**options)
    valid=np.asarray(mask)>0
    result=np.asarray(out).copy()
    result[valid]=np.asarray(original_part)[valid]
    if verify_support:
        support=Image.new('L',extended.size,255).rotate(angle,fillcolor=0,**options).crop(box)
        assert np.all(np.asarray(support)==255),'パディングが不足しています。'
        assert np.array_equal(result[valid],np.asarray(original_part)[valid])
    return Image.fromarray(result)


In [ ]:
TEST_DIR = ROOT/'data/transistor/test'
expected_manifest = read_json(ROTATION_RUN/'test_manifest.json')
assert expected_manifest == read_json(NORMAL_RUN/'test_manifest.json')
paths = sorted(TEST_DIR.glob('*/*.png'),key=lambda p:p.relative_to(TEST_DIR).as_posix())
assert [p.relative_to(ROOT).as_posix() for p in paths] == [r['repo_path'] for r in expected_manifest]
base_images,base_tensors,image_records = [],[],[]
for path,record in zip(paths,expected_manifest):
    raw = path.read_bytes()
    assert hashlib.sha256(raw).hexdigest() == record['file_sha256']
    with Image.open(BytesIO(raw)) as im:
        assert im.size == (1024,1024)
        image = im.convert('RGB').copy()
    tensor = to_tensor(final_transform(image))
    assert hashlib.sha256(tensor.numpy().tobytes()).hexdigest() == record['preprocessed_sha256']
    base_images.append(image);base_tensors.append(tensor);image_records.append(record)
base_tensor = torch.stack(base_tensors)
image_table = pd.DataFrame(image_records)
counts = image_table.category.value_counts().to_dict()
assert counts == {'good':60,'bent_lead':10,'cut_lead':10,'damaged_case':10,'misplaced':10}
manifest_hash = hashlib.sha256(json.dumps(image_records,ensure_ascii=False,sort_keys=True).encode('utf-8')).hexdigest()
assert manifest_hash == normal_config['test_manifest_sha256'] == rotation_config['test_manifest_sha256']
assert np.array_equal(baseline_scores.loc[image_table.relative_path,'true_label'],image_table.true_label)
display(pd.DataFrame([{'種類':CATEGORY_LABELS[k],'枚数':counts[k]} for k in CATEGORY_LABELS]))


assert read_json(KNOWN_DIR/'test_manifest.json') == image_records
assert known_config['test_manifest_sha256'] == manifest_hash
assert not known_assignment.relative_path.duplicated().any()
known_assignment = known_assignment.set_index('relative_path').loc[image_table.relative_path].reset_index()
assert len(known_assignment) == 100
assert np.array_equal(known_assignment.true_label, image_table.true_label)
assert np.isin(known_assignment.assigned_angle, range(36,360,36)).all()
for name in CONDITIONS:
    part = known_scores[known_scores.condition == name].set_index('relative_path').loc[image_table.relative_path]
    assert len(part) == 100 and part.index.is_unique
    assert np.array_equal(part.true_label, image_table.true_label)
    assert np.array_equal(part.category, image_table.category)
    assert np.array_equal(part.angle, known_assignment.assigned_angle)


In [ ]:
class VAE_Hybrid_CNN_FC(nn.Module):
    def __init__(self, latent_dim=64):
        super(VAE_Hybrid_CNN_FC, self).__init__()

        # -------------------------------------------------------------
        # 【エンコーダ前半】 CNN_Deep5（空間圧縮: 128x128 -> 4x4）
        # -------------------------------------------------------------
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(),

            nn.Conv2d(16, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU()
        )

        # -------------------------------------------------------------
        # 【エンコーダ後半】 FC_Only（段階的圧縮: 4096次元 -> 128次元）
        # -------------------------------------------------------------
        self.encoder_fc = nn.Sequential(
            nn.Linear(256 * 4 * 4, 1024), nn.ReLU(),
            nn.Linear(1024, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU()
        )

        # 潜在空間へのマッピング（128次元 -> 64次元）
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

        # -------------------------------------------------------------
        # 【デコーダ前半】 FC_Only（段階的復元: 64次元 -> 4096次元）
        # -------------------------------------------------------------
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, 1024), nn.ReLU(),
            # ※CNNへ繋ぐため、最後はSigmoidではなくReLUを使用します
            nn.Linear(1024, 256 * 4 * 4), nn.ReLU()
        )

        # -------------------------------------------------------------
        # 【デコーダ後半】 CNN_Deep5（空間復元: 4x4 -> 128x128）
        # -------------------------------------------------------------
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(),

            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(),

            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(),

            nn.ConvTranspose2d(16, 1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x, sample=True):
        # 1. CNNによる特徴抽出
        h = self.encoder_conv(x)
        h = torch.flatten(h, start_dim=1)

        # 2. FCによる段階的圧縮
        h = self.encoder_fc(h)

        # 3. 潜在変数の計算と安全装置（クランプ）
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        logvar = torch.clamp(logvar, min=-10.0, max=10.0)

        std = torch.exp(0.5 * logvar)
        z = mu + torch.randn_like(std) * std if sample else mu

        # 4. FCによる段階的復元
        h_recon = self.decoder_fc(z)
        h_recon = h_recon.view(-1, 256, 4, 4) # 4階テンソルに戻す

        # 5. CNNによる画像復元
        recon = self.decoder_conv(h_recon)

        return recon, mu, logvar


In [ ]:
def binary_roc_curve(y_true, scores):
    labels = np.asarray(y_true)
    scores = np.asarray(scores, dtype=float)
    if labels.ndim != 1 or scores.ndim != 1 or labels.shape != scores.shape:
        raise ValueError('ラベルとスコアは同じ長さの1次元配列にしてください。')
    if not np.isfinite(scores).all() or not np.isin(labels, [0, 1]).all():
        raise ValueError('スコアは有限値、ラベルは0または1である必要があります。')
    positives = int((labels == 1).sum())
    negatives = int((labels == 0).sum())
    if not positives or not negatives:
        raise ValueError('正常と異常の両方が必要です。')
    order = np.argsort(-scores, kind='stable')
    ordered_scores = scores[order]
    ordered_labels = labels[order]
    ends = np.r_[np.flatnonzero(np.diff(ordered_scores)), len(scores) - 1]
    true_positives = np.cumsum(ordered_labels)[ends]
    false_positives = (ends + 1) - true_positives
    fpr = np.r_[0.0, false_positives / negatives]
    tpr = np.r_[0.0, true_positives / positives]
    thresholds = np.r_[np.inf, ordered_scores[ends]]
    return fpr, tpr, thresholds

def binary_auroc(y_true, scores):
    fpr, tpr, _ = binary_roc_curve(y_true, scores)
    return float(np.trapezoid(tpr, fpr))

assert binary_auroc([0,0,1,1], [0.1,0.2,0.8,0.9]) == 1.0
assert binary_auroc([0,0,1,1], [0.9,0.8,0.2,0.1]) == 0.0
assert binary_auroc([0,0,1,1], [0.5,0.5,0.5,0.5]) == 0.5



def verify_auroc(y,scores):
    value=binary_auroc(y,scores)
    normal=scores[y==0]; abnormal=scores[y==1]
    rank=((abnormal[:,None]>normal).sum()+0.5*(abnormal[:,None]==normal).sum())/(len(normal)*len(abnormal))
    assert np.isclose(value,rank,atol=1e-12)
    return value


In [ ]:
def score_loaded_model(model,images):
    model.eval()
    reconstructed = np.empty((len(images),128,128),dtype=np.float32)
    error_maps = np.empty_like(reconstructed)
    scores = np.empty(len(images),dtype=np.float64)
    with torch.inference_mode():
        for start in range(0,len(images),EVAL_BATCH_SIZE):
            x = images[start:start+EVAL_BATCH_SIZE].to(DEVICE)
            output,_,_ = model(x,sample=False)
            assert torch.isfinite(output).all()
            for local in range(len(x)):
                index = start+local
                a = x[local,0].cpu().numpy()
                b = output[local,0].cpu().numpy()
                _,ssim_map = ssim(a,b,data_range=1.0,win_size=11,full=True)
                error = 1.0-ssim_map
                assert np.isfinite(error).all()
                reconstructed[index] = b
                error_maps[index] = error
                scores[index] = np.sort(error.ravel())[::-1][:TOP_K].mean()
    return scores,reconstructed,error_maps


In [ ]:
def new_id():
    return datetime.now(ZoneInfo('Asia/Tokyo')).strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]

def object_hash(value):
    return hashlib.sha256(json.dumps(value,ensure_ascii=False,sort_keys=True).encode('utf-8')).hexdigest()

def tensor_hash(tensor):
    return hashlib.sha256(tensor.detach().cpu().contiguous().numpy().tobytes()).hexdigest()

def seed_everything(seed):
    random.seed(int(seed));np.random.seed(int(seed));torch.manual_seed(int(seed))
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(int(seed))

def copy_cpu_state(model):
    return {key:value.detach().cpu().clone() for key,value in model.state_dict().items()}

def state_digest(state):
    h=hashlib.sha256()
    for key,value in state.items():
        h.update(key.encode('utf-8')+str(value.dtype).encode('ascii'))
        h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

def load_old_models():
    loaded={}
    for name,path in MODEL_PATHS.items():
        model=VAE_Hybrid_CNN_FC(latent_dim=64).to(DEVICE)
        model.load_state_dict(torch.load(path,map_location=DEVICE,weights_only=True),strict=True)
        model.eval();model.requires_grad_(False);loaded[name]=model
    return loaded

def rotate_test_inputs(angles,indices=None):
    indices=list(range(len(image_table))) if indices is None else list(indices)
    assert len(angles)==len(indices)
    return torch.stack([to_tensor(final_transform(rotate_with_reflect(base_images[i],float(a),verify_support=True)))
                        for i,a in zip(indices,angles)])

def reconstruction_metrics(inputs,reconstructed):
    mse=np.mean((inputs[:,0].numpy()-reconstructed)**2,axis=(1,2))
    similarity=np.array([ssim(a,b,data_range=1.0,win_size=11)
        for a,b in zip(inputs[:,0].numpy(),reconstructed)])
    return mse,similarity

SESSION_DIR=OUTPUT_BASE/new_id()
SESSION_DIR.mkdir(parents=True,exist_ok=False)
RECHECK_DIR=SESSION_DIR/'01_preprocessing_check';RECHECK_DIR.mkdir()
SWEEP_DIR=SESSION_DIR/'02_angle_sweep';SWEEP_DIR.mkdir()
session_config={'status':'prepared','run_id':SESSION_DIR.name,'created_at_jst':datetime.now(ZoneInfo('Asia/Tokyo')).isoformat(),
    'source_hashes':source_hashes,'device':str(DEVICE),'python':platform.python_version(),
    'versions':{v:importlib.metadata.version(v) for v in ['torch','numpy','pandas','Pillow','scikit-image']},
    'preprocessing':'Pillow direct: RGB1024 / reflect / NEAREST / preserve supported pixels / LANCZOS128 / L / float32 / 255',
    'ssim_window':11,'data_range':1,'top_k':TOP_K,'score_atol':SCORE_ATOL,'score_rtol':SCORE_RTOL,'auroc_atol':AUROC_ATOL}
write_json(SESSION_DIR/'session_config.json',session_config)
write_json(SESSION_DIR/'test_manifest.json',image_records)
print('今回のセッション保存先：',SESSION_DIR)


## 第1部：同じ既知角度で、現在の前処理と前回結果を照合

- 前回の角度割り当てをそのまま再利用。角度を抽選し直しません。
- 入力のfloat32テンソルは**SHA-256完全一致**を要求。
- スコアは `|今回−前回| ≤ 1e-6 + 1e-6 × |前回|` を要求し、最大差も報告。
- AUROCは絶対差1e-12以下を要求。スコアが近くても順位が変われば分かるようにします。
- 全入力・全スコアの照合表を保存してから判定。不一致なら停止します。


In [ ]:
known_input_records=read_json(KNOWN_DIR/'rotated_input_manifest.json')
known_input_by_path={r['relative_path']:r for r in known_input_records}
assert len(known_input_by_path)==100
known_angles=known_assignment.assigned_angle.to_numpy(dtype=int)
known_tensor=rotate_test_inputs(known_angles)
hash_rows=[]
for scenario,inputs in [('unrotated',base_tensor),('known_rotated',known_tensor)]:
    for i,row in image_table.iterrows():
        if scenario=='unrotated':
            expected_hash=image_records[i]['preprocessed_sha256']
        else:
            prior=known_input_by_path[row.relative_path]
            assert prior['angle']==int(known_angles[i])
            assert prior['source_sha256']==image_records[i]['file_sha256']
            expected_hash=prior['preprocessed_sha256']
        current_hash=tensor_hash(inputs[i])
        hash_rows.append({'scenario':scenario,'relative_path':row.relative_path,
            'expected_sha256':expected_hash,'current_sha256':current_hash,'equal':current_hash==expected_hash})
hash_report=pd.DataFrame(hash_rows)
hash_report.to_csv(RECHECK_DIR/'input_hash_comparison.csv',index=False)
np.savez_compressed(RECHECK_DIR/'inputs.npz',unrotated=base_tensor.numpy(),known_rotated=known_tensor.numpy(),
    known_angles=known_angles,relative_paths=image_table.relative_path.to_numpy(dtype=str))
display(hash_report.groupby('scenario').equal.agg(['count','sum']))
INPUT_CHECK_PASSED=bool(hash_report.equal.all())
write_json(RECHECK_DIR/'status.json',{'status':'inputs_checked','input_hashes_equal':INPUT_CHECK_PASSED})
assert INPUT_CHECK_PASSED,'前処理の画素値が前回と不一致です。input_hash_comparison.csvを確認して停止してください。'

old_models=load_old_models()
recheck_rows,recheck_summary=[],[]
try:
    for scenario,inputs in [('unrotated',base_tensor),('known_rotated',known_tensor)]:
        arrays={'relative_paths':image_table.relative_path.to_numpy(dtype=str)}
        for name,model in old_models.items():
            scores,reconstructed,maps=score_loaded_model(model,inputs)
            if scenario=='unrotated':
                prior_scores=baseline_scores.loc[image_table.relative_path,name].to_numpy()
                prior_auc=float(baseline_comparison.loc[name,'auroc'])
            else:
                prior_scores=known_scores[known_scores.condition==name].set_index('relative_path').loc[image_table.relative_path,'score'].to_numpy()
                prior_auc=float(known_comparison[(known_comparison.condition==name)&(known_comparison.scenario=='random_rotated')].iloc[0].auroc)
            equal=np.isclose(scores,prior_scores,rtol=SCORE_RTOL,atol=SCORE_ATOL)
            auc=verify_auroc(image_table.true_label.to_numpy(),scores)
            for i,row in image_table.iterrows():
                recheck_rows.append({'scenario':scenario,'condition':name,'relative_path':row.relative_path,
                    'previous_score':float(prior_scores[i]),'current_score':float(scores[i]),
                    'absolute_difference':float(abs(scores[i]-prior_scores[i])),'within_tolerance':bool(equal[i])})
            recheck_summary.append({'scenario':scenario,'condition':name,'all_scores_close':bool(equal.all()),
                'max_score_difference':float(np.max(np.abs(scores-prior_scores))),'previous_auroc':prior_auc,
                'current_auroc':auc,'auroc_close':bool(np.isclose(auc,prior_auc,rtol=0,atol=AUROC_ATOL))})
            arrays[f'{name}_reconstruction']=reconstructed;arrays[f'{name}_error_map']=maps;arrays[f'{name}_scores']=scores
        np.savez_compressed(RECHECK_DIR/f'{scenario}_results.npz',**arrays)
except BaseException:
    old_models.clear()
    raise
recheck_scores=pd.DataFrame(recheck_rows);recheck_table=pd.DataFrame(recheck_summary)
recheck_scores.to_csv(RECHECK_DIR/'score_comparison.csv',index=False)
recheck_table.to_csv(RECHECK_DIR/'auroc_comparison.csv',index=False)
display(recheck_table.replace({'condition':LABELS,'scenario':SCENARIO_LABELS}))
PREPROCESSING_CHECK_PASSED=bool(recheck_table.all_scores_close.all() and recheck_table.auroc_close.all())
write_json(RECHECK_DIR/'status.json',{'status':'passed' if PREPROCESSING_CHECK_PASSED else 'mismatch',
    'input_hashes_equal':INPUT_CHECK_PASSED,'score_atol':SCORE_ATOL,'score_rtol':SCORE_RTOL,'auroc_atol':AUROC_ATOL})
assert PREPROCESSING_CHECK_PASSED,'スコアまたはAUROCが前回と不一致です。保存した照合表を確認して停止してください。'
print('第1部の照合を通過しました。今回のPillow処理と既存結果が指定範囲で一致しています。')


## 第2部：同じ正常画像の角度を少しずつ変える（再学習なし）

初期表示はファイル名順の正常5枚。設定の `NORMAL_SWEEP_COUNT=None` で全正常60枚。
0・36・72度が学習角度、18・54度が中間、その他も未知角度です。
入力／両モデルの復元／誤差マップを3角度ずつ表示します。
**向きが最も近い学習角度へ偏るという仮説は、観察対象であり定量的に証明済みではありません。**
復元MSE・SSIMは補助情報で、異常検知の性能指標AUROCと区別します。


In [ ]:
assert PREPROCESSING_CHECK_PASSED
normal_indices=image_table.index[image_table.category=='good'].tolist()
assert NORMAL_SWEEP_COUNT is None or (isinstance(NORMAL_SWEEP_COUNT,int) and not isinstance(NORMAL_SWEEP_COUNT,bool) and NORMAL_SWEEP_COUNT>0)
if NORMAL_SWEEP_COUNT is not None: normal_indices=normal_indices[:NORMAL_SWEEP_COUNT]
assert all(0<=a<360 for a in SWEEP_ANGLES)
write_json(SWEEP_DIR/'settings.json',{'normal_paths':image_table.loc[normal_indices,'relative_path'].tolist(),
    'angles':list(SWEEP_ANGLES),'training_angles':list(TRAINING_ANGLES),'preprocessing_check_passed':True})
sweep_rows=[]
for angle in SWEEP_ANGLES:
    inputs=rotate_test_inputs([angle]*len(normal_indices),normal_indices)
    arrays={'inputs':inputs.numpy(),'relative_paths':image_table.loc[normal_indices,'relative_path'].to_numpy(dtype=str)}
    for name,model in old_models.items():
        scores,reconstructed,maps=score_loaded_model(model,inputs)
        mse,similarity=reconstruction_metrics(inputs,reconstructed)
        arrays[f'{name}_reconstruction']=reconstructed;arrays[f'{name}_error_map']=maps
        for j,index in enumerate(normal_indices):
            sweep_rows.append({'condition':name,'relative_path':image_table.loc[index,'relative_path'],
                'angle':angle,'seen_angle':angle in TRAINING_ANGLES,'mse':float(mse[j]),
                'ssim':float(similarity[j]),'top130_score':float(scores[j])})
    np.savez_compressed(SWEEP_DIR/f'angle_{angle:03d}.npz',**arrays)
sweep_table=pd.DataFrame(sweep_rows);sweep_table.to_csv(SWEEP_DIR/'normal_reconstruction_metrics.csv',index=False)
display(sweep_table.replace({'condition':LABELS}))
fig,axes=plt.subplots(1,3,figsize=(16,5),constrained_layout=True)
for name in CONDITIONS:
    means=sweep_table[sweep_table.condition==name].groupby('angle')[['mse','ssim','top130_score']].mean().loc[list(SWEEP_ANGLES)]
    for ax,metric,title in zip(axes,['mse','ssim','top130_score'],['復元MSE（小さいほどよい）','SSIM（大きいほどよい）','Top-130異常スコア']):
        ax.plot(means.index,means[metric],'o-',color=COLORS[name],label=LABELS[name])
        ax.set(xlabel='回転角度（度）',ylabel=metric,title=title,xticks=SWEEP_ANGLES)
        ax.grid(ls='--',alpha=.4);ax.legend()
fig.suptitle(f'正常{len(normal_indices)}枚の平均：学習角度は0・36・72度')
fig.savefig(SWEEP_DIR/'metrics_by_angle.png',dpi=180);plt.show();plt.close(fig)
for j,index in enumerate(normal_indices):
    for start in range(0,len(SWEEP_ANGLES),SWEEP_COLUMNS_PER_FIG):
        chunk=SWEEP_ANGLES[start:start+SWEEP_COLUMNS_PER_FIG]
        fig,axes=plt.subplots(5,len(chunk),figsize=(3.6*len(chunk),12),squeeze=False,constrained_layout=True)
        for col,angle in enumerate(chunk):
            with np.load(SWEEP_DIR/f'angle_{angle:03d}.npz',allow_pickle=False) as cache:
                axes[0,col].imshow(cache['inputs'][j,0],cmap='gray',vmin=0,vmax=1)
                axes[0,col].set_title(f'{angle}度 / '+('学習角度' if angle in TRAINING_ANGLES else '未知角度'))
                for k,name in enumerate(CONDITIONS):
                    axes[1+2*k,col].imshow(cache[f'{name}_reconstruction'][j],cmap='gray',vmin=0,vmax=1)
                    heat=axes[2+2*k,col].imshow(cache[f'{name}_error_map'][j],cmap='inferno',vmin=0,vmax=2)
        for row,label in enumerate(['入力','通常学習：復元','通常学習：誤差','回転拡張：復元','回転拡張：誤差']):
            axes[row,0].set_ylabel(label)
        for ax in axes.flat: ax.set_xticks([]);ax.set_yticks([])
        fig.colorbar(heat,ax=axes[[2,4],:].ravel().tolist(),label='1−SSIM（全図0〜2）',shrink=.65)
        fig.suptitle(image_table.loc[index,'relative_path'])
        fig.savefig(SWEEP_DIR/f'{Path(image_table.loc[index,"relative_path"]).stem}_angles_{chunk[0]:03d}_{chunk[-1]:03d}.png',dpi=150)
        plt.show();plt.close(fig)
old_models.clear()
if 'model' in globals(): del model
print('第2部完了。次の第3部には、新規3モデルの正式学習があります。')


## 第3部：新規3モデルを同じ更新回数・βで比較する

| 条件 | 学習入力 | 1周の延べ画像数 | 総更新回数 |
| --- | --- | ---: | ---: |
| A：通常学習 | 正常画像のみ | 213 | 6,700 |
| B：固定回転 | 0〜324度の36度刻み | 2,130 | 6,700 |
| C：ランダム回転 | 画像を使うたびに $[0,360)$ から連続角度を等確率で抽選 | 2,130 | 6,700 |

B/Cは同じ元画像の10枠を同じ順にシャッフルする。Cの枠番号は角度を固定しない。
Cでは画像読み込み用の専用乱数を使い、潜在変数のサンプリング用乱数と分離する。
正規化前の高解像度画像を毎回回転するため、特にCの学習には時間がかかります。

学習損失は **SSE＋β×潜在次元で合計した標準正規KL**（いずれもバッチも合計）。
ログは画像枚数で割って表示。MSEは学習後の正常復元の補助評価に使います。

$$\beta(t)=\min\left(1,\frac{t-1}{3350-1}\right),\quad t=1,\ldots,6700$$

最初の更新で0、3,350更新目で1、その後1。3条件で同じです。
過去モデルとはβの進み方が異なるため、角度の選び方の効果は**今回のB/C間**で比較します。
AとB/Cには端数バッチ・総提示画像数・順序・BatchNorm統計の違いが残ります。
単一学習seed42で、テスト結果によるモデル・エポック選択はしません。

### 保存済みの新規学習モデルから評価を再開する場合

設定セルで `RUN_FULL_TRAINING=False`、`TRAINING_RUN_ID='完了したセッションのフォルダ名'` にして、
共通準備と第1・2部、以下の定義セルを実行してください。学習セルはモデルを読み込むだけになります。
`TRAINING_RUN_ID=None` の場合は、設定が一致する完了済み実行のうち最新を選びます。
初期値は正式学習を行う `True` です。途中で停止した学習の継続は行わず、新しく開始します。


In [ ]:
assert PREPROCESSING_CHECK_PASSED
# 長時間の学習前に、後で共用する保存済み入力の存在を確認する。
for filename in ['config.json','test_manifest.json','angle_assignments.csv',
                 'rotated_input_manifest.json','output_file_hashes.json']:
    existing=UNSEEN_RUN/filename
    if not existing.is_file(): raise FileNotFoundError(existing)
    source_hashes[existing.relative_to(ROOT).as_posix()]=sha256(existing)
for scenario in ANGLE_CANDIDATES:
    for assignment_seed in ASSIGNMENT_SEEDS:
        existing=UNSEEN_RUN/f'cases/{scenario}_seed{assignment_seed}/evaluation_arrays.npz'
        if not existing.is_file(): raise FileNotFoundError(existing)
train_manifest=read_json(ROTATION_RUN/'train_manifest.json')
train_paths=sorted((ROOT/'data/transistor/train/good').glob('*.png'),key=lambda p:p.name)
assert len(train_paths)==len(train_manifest)==213
assert [p.relative_to(ROOT).as_posix() for p in train_paths]==[r['path'] for r in train_manifest]
train_images,train_base=[] , []
for path,record in zip(train_paths,train_manifest):
    raw=path.read_bytes();assert hashlib.sha256(raw).hexdigest()==record['file_sha256']
    with Image.open(BytesIO(raw)) as im:
        assert im.size==(1024,1024)
        image=im.convert('RGB').copy()
    tensor=to_tensor(final_transform(image));assert tensor_hash(tensor)==record['preprocessed_sha256']
    train_images.append(image);train_base.append(tensor)
train_base=torch.stack(train_base)
for path in [ROTATION_RUN/'train_manifest.json',KNOWN_DIR/'rotated_input_manifest.json']:
    source_hashes[path.relative_to(ROOT).as_posix()]=sha256(path)
def beta_at_update(update):
    if not 1<=update<=TARGET_UPDATES: raise ValueError('更新番号が範囲外です。')
    return float(BETA_MAX*min(1.0,(update-1)/(WARMUP_UPDATES-1)))
assert beta_at_update(1)==0 and beta_at_update(3350)==1 and beta_at_update(6700)==1
beta_schedule=np.array([beta_at_update(t) for t in range(1,TARGET_UPDATES+1)])
TRAIN_CONTRACT={'model':'Hybrid CNN + FC','model_definition_sha256':MODEL_DEFINITION_SHA,
    'seed':SEED,'latent_dim':LATENT_DIM,'batch_size':BATCH_SIZE,'target_updates':TARGET_UPDATES,
    'optimizer':'Adam','learning_rate':LEARNING_RATE,'adam_betas':[0.9,0.999],'adam_eps':1e-8,'weight_decay':0,
    'loss':'pixel SSE sum + beta * standard normal KL latent sum; batch sum',
    'beta_rule':'min(1,(one_based_update-1)/(3350-1))','beta_max':BETA_MAX,'warmup_updates':WARMUP_UPDATES,
    'beta_schedule_sha256':hashlib.sha256(beta_schedule.tobytes()).hexdigest(),
    'train_manifest_sha256':object_hash(train_manifest),'test_manifest_sha256':manifest_hash,
    'train_conditions':list(TRAIN_CONDITIONS),'fixed_angles':list(TRAINING_ANGLES),
    'random_angle':'independent uniform float in [0,360) on every __getitem__; private NumPy RNG',
    'sampler_seed':SEED+1000,'augmentation_seed':SEED+2000,'workers':0,'preprocessing':session_config['preprocessing']}
CONTRACT_SHA=object_hash(TRAIN_CONTRACT)

class TrainingImages(Dataset):
    def __init__(self,condition,fixed_cache=None):
        self.condition=condition;self.repeats=1 if condition=='A_normal' else 10
        self.fixed_cache=fixed_cache
        self.angle_rng=np.random.default_rng(SEED+2000)
    def __len__(self): return len(train_images)*self.repeats
    def __getitem__(self,index):
        parent,variant=divmod(int(index),self.repeats)
        if self.condition=='A_normal': tensor=train_base[parent];angle=0.0
        elif self.condition=='B_fixed': tensor=self.fixed_cache[index];angle=float(TRAINING_ANGLES[variant])
        else:
            angle=float(self.angle_rng.uniform(0.0,360.0))
            tensor=to_tensor(final_transform(rotate_with_reflect(train_images[parent],angle)))
        return tensor,parent,variant,angle,int(index)

def training_loss(output,inputs,mu,logvar,beta):
    sse=(output-inputs).square().sum()
    kl=-0.5*(1+logvar-mu.square()-logvar.exp()).sum()
    return sse,kl,sse+beta*kl

def train_condition(condition,directory,initial,fixed_cache):
    directory.mkdir(exist_ok=False)
    dataset=TrainingImages(condition,fixed_cache)
    generator=torch.Generator().manual_seed(SEED+1000)
    loader=DataLoader(dataset,batch_size=BATCH_SIZE,shuffle=True,generator=generator,
        drop_last=False,num_workers=0)
    seed_everything(SEED)
    model=VAE_Hybrid_CNN_FC(latent_dim=LATENT_DIM).to(DEVICE)
    model.load_state_dict(initial,strict=True)
    initial_hash=state_digest(copy_cpu_state(model))
    # 初期モデル構築による乱数消費を戻し、潜在サンプル列を共通にする。
    seed_everything(SEED)
    optimizer=torch.optim.Adam(model.parameters(),lr=LEARNING_RATE)
    steps,windows=[],[];window_sums=np.zeros(3);window_seen=0
    order_hash=hashlib.sha256();angle_hash=hashlib.sha256()
    update=0;epoch=0;started=time.perf_counter()
    config={'status':'running','condition':condition,'contract':TRAIN_CONTRACT,'contract_sha256':CONTRACT_SHA,
        'initial_state_sha256':initial_hash,'images_per_epoch':len(dataset),
        'updates_per_epoch':math.ceil(len(dataset)/BATCH_SIZE)}
    write_json(directory/'config.json',config)
    try:
        with (directory/'sample_trace.csv').open('w',encoding='utf-8',newline='') as stream:
            writer=csv.writer(stream);writer.writerow(['update','parent_index','variant','dataset_index','angle'])
            while update<TARGET_UPDATES:
                epoch+=1
                for batch_index,(images,parents,variants,angles,indices) in enumerate(loader,1):
                    update+=1;beta=beta_at_update(update)
                    images=images.to(DEVICE);model.train();optimizer.zero_grad(set_to_none=True)
                    output,mu,logvar=model(images,sample=True)
                    sse,kl,total=training_loss(output,images,mu,logvar,beta)
                    if not all(torch.isfinite(v).item() for v in [sse,kl,total]):
                        raise FloatingPointError('損失にNaN/infがあります。')
                    total.backward()
                    if any(p.grad is not None and not torch.isfinite(p.grad).all().item() for p in model.parameters()):
                        raise FloatingPointError('勾配にNaN/infがあります。')
                    optimizer.step()
                    n=len(images);values=np.array([sse.item(),kl.item(),total.item()])
                    window_sums+=values;window_seen+=n
                    order_hash.update(indices.numpy().astype('<i8').tobytes())
                    angle_hash.update(angles.numpy().astype('<f8').tobytes())
                    writer.writerows((update,int(parent),int(variant),int(index),repr(float(angle)))
                        for parent,variant,index,angle in zip(parents,variants,indices,angles))
                    steps.append({'update':update,'epoch':epoch,'batch_in_epoch':batch_index,'batch_size':n,
                        'beta':beta,'sse_per_image':values[0]/n,'kl_per_image':values[1]/n,'total_per_image':values[2]/n})
                    if update%67==0 or update==TARGET_UPDATES:
                        windows.append({'update':update,'seen_images':window_seen,
                            'sse_per_image':window_sums[0]/window_seen,'kl_per_image':window_sums[1]/window_seen,
                            'total_per_image':window_sums[2]/window_seen})
                        window_sums[:]=0;window_seen=0;stream.flush()
                        pd.DataFrame(steps).to_csv(directory/'step_history.csv',index=False)
                        pd.DataFrame(windows).to_csv(directory/'history.csv',index=False)
                        if update==67 or update%670==0:
                            print(f'{TRAIN_LABELS[condition]} / {update:,}/{TARGET_UPDATES:,}更新 / β={beta:.4f}',flush=True)
                    if update==TARGET_UPDATES: break
        torch.save(copy_cpu_state(model),directory/'model_final.pth')
        summary={'status':'completed','condition':condition,'updates':update,'epochs_started':epoch,
            'seen_images':int(sum(r['batch_size'] for r in steps)),'seconds':time.perf_counter()-started,
            'initial_state_sha256':initial_hash,'model_sha256':sha256(directory/'model_final.pth'),
            'sample_order_sha256':order_hash.hexdigest(),'angle_sequence_sha256':angle_hash.hexdigest(),
            'contract_sha256':CONTRACT_SHA}
        write_json(directory/'summary.json',summary)
        config.update(status='completed');write_json(directory/'config.json',config)
        return summary
    except BaseException as exc:
        pd.DataFrame(steps).to_csv(directory/'step_history.csv',index=False)
        config.update(status='failed_or_interrupted',error=f'{type(exc).__name__}: {exc}',updates=update)
        write_json(directory/'config.json',config);raise
    finally:
        del model,optimizer

display(pd.DataFrame([{'条件':TRAIN_LABELS[k],'1周の延べ枚数':213 if k=='A_normal' else 2130,
    '更新回数':TARGET_UPDATES,'損失':'SSE + β KL'} for k in TRAIN_CONDITIONS]))


In [ ]:
# 正式学習セル。初期値True。第1・2部だけ行う場合はこのセルより前で止める。
assert PREPROCESSING_CHECK_PASSED
if RUN_FULL_TRAINING:
    TRAINING_DIR=SESSION_DIR/'training';TRAINING_DIR.mkdir(exist_ok=False)
    run_config={'status':'running','contract':TRAIN_CONTRACT,'contract_sha256':CONTRACT_SHA,
        'source_hashes':source_hashes,'versions':session_config['versions'],'run_id':SESSION_DIR.name}
    write_json(TRAINING_DIR/'config.json',run_config)
    write_json(TRAINING_DIR/'train_manifest.json',train_manifest)
    summaries={};fixed_cache=None
    try:
        seed_everything(SEED);initial_model=VAE_Hybrid_CNN_FC(latent_dim=LATENT_DIM)
        initial=copy_cpu_state(initial_model);del initial_model
        torch.save(initial,TRAINING_DIR/'initial_state.pth')
        for condition in TRAIN_CONDITIONS:
            if condition=='B_fixed':
                print('Bの固定回転2,130枚を高解像度から準備しています。',flush=True)
                fixed_cache=torch.stack([to_tensor(final_transform(rotate_with_reflect(image,float(angle))))
                    for image in train_images for angle in TRAINING_ANGLES])
                write_json(TRAINING_DIR/'fixed_input_hash.json',{'sha256':tensor_hash(fixed_cache)})
            summaries[condition]=train_condition(condition,TRAINING_DIR/condition,initial,fixed_cache)
            # Cは毎回回転するため、固定入力キャッシュは以降使用しない。
            if condition=='B_fixed': fixed_cache=None
            pd.DataFrame(summaries.values()).to_csv(TRAINING_DIR/'training_summary.csv',index=False)
        assert summaries['B_fixed']['sample_order_sha256']==summaries['C_random']['sample_order_sha256']
        run_config.update(status='completed');write_json(TRAINING_DIR/'config.json',run_config)
    except BaseException as exc:
        run_config.update(status='failed_or_interrupted',error=f'{type(exc).__name__}: {exc}')
        write_json(TRAINING_DIR/'config.json',run_config);raise
    finally:
        del fixed_cache
else:
    if TRAINING_RUN_ID is not None:
        if Path(TRAINING_RUN_ID).name!=TRAINING_RUN_ID: raise ValueError('実行フォルダ名だけを指定してください。')
        TRAINING_DIR=OUTPUT_BASE/TRAINING_RUN_ID/'training'
    else:
        candidates=[]
        for path in OUTPUT_BASE.glob('*/training/config.json'):
            saved=read_json(path)
            if saved.get('status')=='completed' and saved.get('contract')==TRAIN_CONTRACT:
                candidates.append(path.parent)
        if not candidates: raise FileNotFoundError('同じ条件の完了済み学習がありません。実行IDを指定してください。')
        TRAINING_DIR=sorted(candidates,key=lambda p:p.parent.name)[-1]
    run_config=read_json(TRAINING_DIR/'config.json')
    if run_config.get('status')!='completed' or run_config.get('contract')!=TRAIN_CONTRACT:
        raise ValueError('未完了、または現在の学習条件と異なる保存結果です。')
    summaries={k:read_json(TRAINING_DIR/k/'summary.json') for k in TRAIN_CONDITIONS}
for condition in TRAIN_CONDITIONS:
    summary=summaries[condition]
    assert summary['status']=='completed' and summary['updates']==TARGET_UPDATES
    assert summary['contract_sha256']==CONTRACT_SHA
    assert sha256(TRAINING_DIR/condition/'model_final.pth')==summary['model_sha256']
    history=pd.read_csv(TRAINING_DIR/condition/'step_history.csv')
    assert np.array_equal(history['update'].to_numpy(),np.arange(1,TARGET_UPDATES+1))
    assert np.allclose(history.beta,beta_schedule,rtol=0,atol=1e-12)
assert len({s['initial_state_sha256'] for s in summaries.values()})==1
assert summaries['B_fixed']['sample_order_sha256']==summaries['C_random']['sample_order_sha256']
display(pd.DataFrame(summaries.values()))
print('新規学習モデルの保存先：',TRAINING_DIR)


### 学習曲線（横軸は更新回数）

3条件の学習入力は異なるため、学習誤差を同一画像上の復元品質とは扱いません。
67更新ごとの画像枚数で重み付けした平均です。正常学習では1エポック平均とは異なります。
KLの対数表示では、浮動小数点の微小な負値・0を描画上だけ1e-12へ置きます（学習損失は変更しません）。


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(16,5),constrained_layout=True)
for condition in TRAIN_CONDITIONS:
    h=pd.read_csv(TRAINING_DIR/condition/'history.csv')
    for ax,key,title in zip(axes,['sse_per_image','kl_per_image','total_per_image'],['復元SSE','KL項','総損失']):
        ax.plot(h['update'],np.maximum(h[key],1e-12),color=TRAIN_COLORS[condition],label=TRAIN_LABELS[condition])
        ax.set(title=title,xlabel='更新回数',ylabel='学習画像1枚あたり',yscale='log');ax.grid(ls='--',alpha=.4);ax.legend()
fig.savefig(SESSION_DIR/'training_curves.png',dpi=180);plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(10,4),constrained_layout=True)
ax.plot(np.arange(1,TARGET_UPDATES+1),beta_schedule,color='teal')
ax.set(title='新規3条件で共通の線形KLアニーリング',xlabel='更新回数',ylabel='β');ax.grid(ls='--',alpha=.4)
fig.savefig(SESSION_DIR/'beta_schedule.png',dpi=180);plt.show();plt.close(fig)


## 新規モデルの評価：回転なし／既知角度／中間角度／未知ランダム角度

回転なし・既知角度は第1部で照合した同じ入力。中間・未知ランダムは11月17日の保存済み10ケースの入力を共用します。
未知角度の配列ファイル・元画像・角度割り当て・入力ハッシュを照合し、角度を引き直しません。
3モデルとも平均μから復元し、SSIM誤差上位130点平均によるAUROCのみで評価。閾値探索なし。
回転なし／既知角度は各1回、未知角度は各5回。5回の標準偏差は角度割り当ての変動です。
ここで使うテスト画像はモデル選択・β調整に使わず、固定した最終重みで比較します。


In [ ]:
unseen_config=read_json(UNSEEN_RUN/'config.json')
assert unseen_config['status']=='completed' and unseen_config['model_hashes']==EXPECTED_HASHES
assert unseen_config['test_manifest_sha256']==manifest_hash
assert read_json(UNSEEN_RUN/'test_manifest.json')==image_records
assert unseen_config['assignment_seeds']==list(ASSIGNMENT_SEEDS)
assert unseen_config['angle_candidates']=={k:list(v) for k,v in ANGLE_CANDIDATES.items()}
saved_assignments=pd.read_csv(UNSEEN_RUN/'angle_assignments.csv')
saved_input_manifest=read_json(UNSEEN_RUN/'rotated_input_manifest.json')
saved_input_hashes={(r['scenario'],r['assignment_seed'],r['relative_path']):r for r in saved_input_manifest}
saved_file_hashes=read_json(UNSEEN_RUN/'output_file_hashes.json')
test_cases=[('unrotated',-1),('known_rotated',42)]+[(scenario,seed) for scenario in ANGLE_CANDIDATES for seed in ASSIGNMENT_SEEDS]
def load_case_inputs(scenario,seed):
    if scenario in ['unrotated','known_rotated']:
        with np.load(RECHECK_DIR/'inputs.npz',allow_pickle=False) as cache:
            assert np.array_equal(cache['relative_paths'],image_table.relative_path.to_numpy(dtype=str))
            inputs=torch.from_numpy(cache[scenario].copy())
            angles=np.zeros(100) if scenario=='unrotated' else cache['known_angles'].copy()
        return inputs,angles
    relative=f'cases/{scenario}_seed{seed}/evaluation_arrays.npz'
    path=UNSEEN_RUN/relative
    assert sha256(path)==saved_file_hashes[relative]
    source_hashes[path.relative_to(ROOT).as_posix()]=sha256(path)
    with np.load(path,allow_pickle=False) as cache:
        assert np.array_equal(cache['relative_paths'],image_table.relative_path.to_numpy(dtype=str))
        inputs=torch.from_numpy(cache['rotated_inputs'].copy());angles=cache['angles'].copy()
    assignment=saved_assignments[(saved_assignments.scenario==scenario)&(saved_assignments.assignment_seed==seed)].set_index('relative_path').loc[image_table.relative_path]
    assert len(assignment)==100 and np.array_equal(angles,assignment.assigned_angle)
    assert np.isin(angles,ANGLE_CANDIDATES[scenario]).all()
    for i,row in image_table.iterrows():
        r=saved_input_hashes[(scenario,seed,row.relative_path)]
        assert r['angle']==int(angles[i]) and r['source_sha256']==image_records[i]['file_sha256']
        assert tensor_hash(inputs[i])==r['preprocessed_sha256']
    return inputs,angles

EVALUATION_DIR=SESSION_DIR/'03_new_model_evaluation';EVALUATION_DIR.mkdir(exist_ok=False)
evaluation_config={'status':'running','training_dir':TRAINING_DIR.relative_to(ROOT).as_posix(),
    'training_contract_sha256':CONTRACT_SHA,'model_hashes':{k:summaries[k]['model_sha256'] for k in TRAIN_CONDITIONS},
    'unseen_source':UNSEEN_RUN.relative_to(ROOT).as_posix(),'test_manifest_sha256':manifest_hash,
    'test_cases':test_cases,'inference':'posterior mean','metric':'AUROC','top_k':TOP_K,'window':11,
    'representative_seed':REPRESENTATIVE_SEED,'versions':session_config['versions']}
write_json(EVALUATION_DIR/'config.json',evaluation_config)
new_models={};score_rows=[];auc_rows=[]
try:
    for condition in TRAIN_CONDITIONS:
        model=VAE_Hybrid_CNN_FC(latent_dim=LATENT_DIM).to(DEVICE)
        model.load_state_dict(torch.load(TRAINING_DIR/condition/'model_final.pth',map_location=DEVICE,weights_only=True),strict=True)
        model.eval();model.requires_grad_(False);new_models[condition]=model
    for scenario,seed in test_cases:
        inputs,angles=load_case_inputs(scenario,seed)
        assert tuple(inputs.shape)==(100,1,128,128)
        arrays={'relative_paths':image_table.relative_path.to_numpy(dtype=str),'angles':angles}
        for condition,model in new_models.items():
            scores,reconstructed,maps=score_loaded_model(model,inputs)
            mse,similarity=reconstruction_metrics(inputs,reconstructed)
            auc=verify_auroc(image_table.true_label.to_numpy(),scores)
            auc_rows.append({'condition':condition,'scenario':scenario,'assignment_seed':seed,'sample_count':100,'auroc':auc})
            arrays[f'{condition}_reconstruction']=reconstructed;arrays[f'{condition}_error_map']=maps
            arrays[f'{condition}_scores']=scores
            for i,row in image_table.iterrows():
                score_rows.append({'condition':condition,'scenario':scenario,'assignment_seed':seed,
                    'relative_path':row.relative_path,'category':row.category,'category_ja':row.category_ja,
                    'true_label':int(row.true_label),'angle':float(angles[i]),'score':float(scores[i]),
                    'mse':float(mse[i]),'ssim':float(similarity[i])})
            print(f'{TRAIN_LABELS[condition]} / {SCENARIO_LABELS[scenario]} / seed{seed} : AUROC={auc:.4f}',flush=True)
        np.savez_compressed(EVALUATION_DIR/f'{scenario}_seed{seed}.npz',**arrays)
        pd.DataFrame(score_rows).to_csv(EVALUATION_DIR/'image_scores.csv',index=False)
        pd.DataFrame(auc_rows).to_csv(EVALUATION_DIR/'trial_auroc.csv',index=False)
        del arrays,inputs,reconstructed,maps,scores
    evaluation_config.update(status='inference_completed')
    write_json(EVALUATION_DIR/'config.json',evaluation_config)
except BaseException as exc:
    evaluation_config.update(status='failed_or_interrupted',error=f'{type(exc).__name__}: {exc}')
    write_json(EVALUATION_DIR/'config.json',evaluation_config);raise
finally:
    new_models.clear()
    if 'model' in globals(): del model
scores_table=pd.DataFrame(score_rows);trial_auroc=pd.DataFrame(auc_rows)
assert len(scores_table)==3600 and len(trial_auroc)==36
assert not scores_table.duplicated(['condition','scenario','assignment_seed','relative_path']).any()


### AUROCと正常復元品質の集計

各100枚で算出したAUROCを集計します。5回のスコアを500枚分としてプールしません。
モデル差は、同じ角度割り当てに対する **C（ランダム）−B（固定）**。
正常画像のMSE・SSIMは各回60枚で平均してから集計し、異常検知AUROCと区別します。


In [ ]:
auc_summary=trial_auroc.groupby(['scenario','condition'],sort=False).agg(
    auroc_mean=('auroc','mean'),auroc_std=('auroc','std'),assignment_count=('auroc','size')).reset_index()
display(auc_summary.replace({'condition':TRAIN_LABELS,'scenario':SCENARIO_LABELS}))
auc_summary.to_csv(EVALUATION_DIR/'auroc_summary.csv',index=False)
paired=trial_auroc.pivot(index=['scenario','assignment_seed'],columns='condition',values='auroc')
paired['C_minus_B']=paired['C_random']-paired['B_fixed']
paired.to_csv(EVALUATION_DIR/'paired_differences.csv');display(paired.rename(columns=TRAIN_LABELS))
paired_summary=paired.groupby('scenario',sort=False).agg(
    mean_C_minus_B=('C_minus_B','mean'),std_C_minus_B=('C_minus_B','std'),
    assignment_count=('C_minus_B','size')).reset_index()
paired_summary.to_csv(EVALUATION_DIR/'paired_difference_summary.csv',index=False)
display(paired_summary.replace({'scenario':SCENARIO_LABELS}))
normal_trial=scores_table[scores_table.category=='good'].groupby(['scenario','assignment_seed','condition']).agg(
    mse=('mse','mean'),ssim=('ssim','mean'),mean_anomaly_score=('score','mean'),normal_count=('score','size')).reset_index()
assert (normal_trial.normal_count==60).all()
normal_summary=normal_trial.groupby(['scenario','condition'],sort=False).agg(
    mse_mean=('mse','mean'),mse_std=('mse','std'),ssim_mean=('ssim','mean'),ssim_std=('ssim','std')).reset_index()
normal_trial.to_csv(EVALUATION_DIR/'normal_reconstruction_by_trial.csv',index=False)
normal_summary.to_csv(EVALUATION_DIR/'normal_reconstruction_summary.csv',index=False)
display(normal_summary.replace({'condition':TRAIN_LABELS,'scenario':SCENARIO_LABELS}))
category_rows=[]
for (scenario,seed,condition),part in scores_table.groupby(['scenario','assignment_seed','condition']):
    for category in CATEGORY_LABELS:
        if category=='good':continue
        subset=part[part.category.isin(['good',category])]
        assert len(subset)==70
        category_rows.append({'scenario':scenario,'assignment_seed':seed,'condition':condition,'category':category,
            'category_ja':CATEGORY_LABELS[category],'auroc':verify_auroc(subset.true_label.to_numpy(),subset.score.to_numpy())})
category_trials=pd.DataFrame(category_rows)
category_summary=category_trials.groupby(['scenario','condition','category_ja'],sort=False).agg(
    auroc_mean=('auroc','mean'),auroc_std=('auroc','std'),assignment_count=('auroc','size')).reset_index()
category_trials.to_csv(EVALUATION_DIR/'category_trial_auroc.csv',index=False)
category_summary.to_csv(EVALUATION_DIR/'category_auroc_summary.csv',index=False)
display(category_summary.replace({'condition':TRAIN_LABELS,'scenario':SCENARIO_LABELS}))
fig,ax=plt.subplots(figsize=(13,6),constrained_layout=True);x=np.arange(4);width=.25
for offset,condition in enumerate(TRAIN_CONDITIONS):
    part=auc_summary[auc_summary.condition==condition].set_index('scenario').loc[list(SCENARIO_LABELS)]
    positions=x+(offset-1)*width;ax.bar(positions,part.auroc_mean,width,label=TRAIN_LABELS[condition],color=TRAIN_COLORS[condition])
    has_std=part.auroc_std.notna().to_numpy()
    ax.errorbar(positions[has_std],part.auroc_mean.to_numpy()[has_std],yerr=part.auroc_std.to_numpy()[has_std],fmt='none',ecolor='black',capsize=4)
    for px,value,std in zip(positions,part.auroc_mean,part.auroc_std.fillna(0)):
        ax.text(px,value+std+.015,f'{value:.4f}',ha='center',fontsize=9)
new_labels=['回転なし','既知角度（1回）','中間角度（5回）','未知ランダム（5回）']
ax.set(xticks=x,xticklabels=new_labels,ylim=(0,1.15),ylabel='AUROC',title='新規3モデル：未知角度は平均±標準偏差')
ax.legend();ax.grid(axis='y',ls='--',alpha=.4);ax.set_axisbelow(True)
fig.savefig(EVALUATION_DIR/'auroc_comparison.png',dpi=180);plt.show();plt.close(fig)
fig,axes=plt.subplots(1,2,figsize=(13,5),constrained_layout=True)
for ax,key,title in zip(axes,['mse_mean','ssim_mean'],['正常60枚の復元MSE','正常60枚のSSIM']):
    for condition in TRAIN_CONDITIONS:
        part=normal_summary[normal_summary.condition==condition].set_index('scenario').loc[list(SCENARIO_LABELS)]
        ax.plot(x,part[key],'o-',label=TRAIN_LABELS[condition],color=TRAIN_COLORS[condition])
    ax.set(xticks=x,xticklabels=new_labels,title=title);ax.tick_params(axis='x',labelrotation=15);ax.grid(ls='--',alpha=.4);ax.legend()
fig.savefig(EVALUATION_DIR/'normal_reconstruction.png',dpi=180);plt.show();plt.close(fig)


### ROC・スコア分布・異常種類別比較

未知角度の代表回は事前指定のseed42。ROCは右下の凡例・破線グリッド・対角線なし。
ヒストグラムは正常＝緑、異常＝赤、40点の共通ビン境界。閾値は探索しません。
種類別は正常60枚と各異常10枚で比較します。少数の異常である点に注意してください。


In [ ]:
for scenario in SCENARIO_LABELS:
    seed=-1 if scenario=='unrotated' else REPRESENTATIVE_SEED
    selected=scores_table[(scores_table.scenario==scenario)&(scores_table.assignment_seed==seed)]
    fig,ax=plt.subplots(figsize=(8,8),constrained_layout=True)
    for condition in TRAIN_CONDITIONS:
        part=selected[selected.condition==condition]
        fpr,tpr,_=binary_roc_curve(part.true_label.to_numpy(),part.score.to_numpy())
        auc=verify_auroc(part.true_label.to_numpy(),part.score.to_numpy())
        ax.plot(fpr,tpr,lw=2,color=TRAIN_COLORS[condition],label=f'{TRAIN_LABELS[condition]} (AUROC={auc:.4f})')
    title={'unrotated':'回転なし','known_rotated':'既知角度'}.get(scenario,SCENARIO_LABELS[scenario])
    ax.set(xlim=(0,1),ylim=(0,1.05),xlabel='False Positive Rate（正常を誤検出した割合）',ylabel='True Positive Rate（異常を検出した割合）',title=f'ROC：{title} / 代表seed{seed}')
    ax.legend(loc='lower right');ax.grid(ls='--',alpha=.5)
    fig.savefig(EVALUATION_DIR/f'roc_{scenario}.png',dpi=180);plt.show();plt.close(fig)
    lo,hi=float(selected.score.min()),float(selected.score.max())
    if lo==hi:lo-=1e-6;hi+=1e-6
    bins=np.linspace(lo,hi,40)
    ymax=max(np.histogram(selected.loc[(selected.condition==condition)&(selected.true_label==label),'score'],bins=bins)[0].max()
        for condition in TRAIN_CONDITIONS for label in [0,1])+1
    for condition in TRAIN_CONDITIONS:
        part=selected[selected.condition==condition];fig,ax=plt.subplots(figsize=(10,6),constrained_layout=True)
        for label,color,word in [(0,'green','正常'),(1,'red','異常')]:
            ax.hist(part.loc[part.true_label==label,'score'],bins=bins,alpha=.5,color=color,edgecolor='white',label=word)
        ax.set(xlim=(lo,hi),ylim=(0,ymax),xlabel='SSIM誤差上位130点の平均',ylabel='画像枚数',title=f'{title} / {TRAIN_LABELS[condition]}')
        ax.legend();ax.grid(axis='y',ls='--',alpha=.4)
        fig.savefig(EVALUATION_DIR/f'hist_{scenario}_{condition}.png',dpi=180);plt.show();plt.close(fig)
    categories=[CATEGORY_LABELS[c] for c in CATEGORY_LABELS if c!='good'];fig,ax=plt.subplots(figsize=(12,6),constrained_layout=True)
    for offset,condition in enumerate(TRAIN_CONDITIONS):
        part=category_summary[(category_summary.scenario==scenario)&(category_summary.condition==condition)].set_index('category_ja').loc[categories]
        ax.bar(np.arange(4)+(offset-1)*.25,part.auroc_mean,width=.25,yerr=part.auroc_std.fillna(0),capsize=3,label=TRAIN_LABELS[condition],color=TRAIN_COLORS[condition])
    ax.set(xticks=np.arange(4),xticklabels=categories,ylim=(0,1.15),ylabel='AUROC',title=f'種類別比較：{title}（未知角度は平均±標準偏差）')
    ax.legend();ax.grid(axis='y',ls='--',alpha=.4);ax.set_axisbelow(True)
    fig.savefig(EVALUATION_DIR/f'category_{scenario}.png',dpi=180);plt.show();plt.close(fig)


### 新規3モデルの入力・復元・誤差マップ

初期表示は各種類の先頭1画像、4条件の代表回です。`VISUAL_PATHS` または `MAX_IMAGES_PER_CATEGORY` で変更できます。
`MAX_IMAGES_PER_CATEGORY=None` なら全100枚になります。設定変更後、この表示セルを再実行するだけで切り替えられます。
未知角度で別の回も見る場合は `EVAL_VISUAL_CASES` に `('midpoint',43)` などを追加してください。
評価時に保存した復元・マップを使うため、表示時にモデルを再推論しません。


In [ ]:
EVAL_VISUAL_CASES=[('unrotated',-1),('known_rotated',42),('midpoint',42),('unseen_random',42)]
if VISUAL_PATHS is not None:
    assert set(VISUAL_PATHS).issubset(set(image_table.relative_path))
    visual_indices=image_table.index[image_table.relative_path.isin(VISUAL_PATHS)].tolist()
else:
    assert MAX_IMAGES_PER_CATEGORY is None or (isinstance(MAX_IMAGES_PER_CATEGORY,int) and MAX_IMAGES_PER_CATEGORY>0)
    visual_indices=[]
    for category in CATEGORY_LABELS:
        indices=image_table.index[image_table.category==category].tolist()
        visual_indices.extend(indices if MAX_IMAGES_PER_CATEGORY is None else indices[:MAX_IMAGES_PER_CATEGORY])
visual_dir=EVALUATION_DIR/'visualizations';visual_dir.mkdir(exist_ok=True)
for scenario,seed in EVAL_VISUAL_CASES:
    assert (scenario,seed) in test_cases
    inputs,angles=load_case_inputs(scenario,seed)
    with np.load(EVALUATION_DIR/f'{scenario}_seed{seed}.npz',allow_pickle=False) as stored:
        arrays={k:stored[k] for k in stored.files}
    assert np.array_equal(arrays['relative_paths'],image_table.relative_path.to_numpy(dtype=str))
    assert np.array_equal(arrays['angles'],angles)
    for index in visual_indices:
        row=image_table.loc[index];fig,axes=plt.subplots(3,4,figsize=(14,10),constrained_layout=True)
        for r,condition in enumerate(TRAIN_CONDITIONS):
            axes[r,0].imshow(base_tensor[index,0],cmap='gray',vmin=0,vmax=1)
            axes[r,1].imshow(inputs[index,0],cmap='gray',vmin=0,vmax=1)
            axes[r,2].imshow(arrays[f'{condition}_reconstruction'][index],cmap='gray',vmin=0,vmax=1)
            heat=axes[r,3].imshow(arrays[f'{condition}_error_map'][index],cmap='inferno',vmin=0,vmax=2)
            axes[r,0].set_ylabel(TRAIN_LABELS[condition])
            for col,title in enumerate(['回転前の入力',f'入力 {angles[index]:g}度',f'復元 / スコア={arrays[f"{condition}_scores"][index]:.4f}','SSIM誤差マップ']):
                axes[r,col].set_title(title,fontsize=10);axes[r,col].set_xticks([]);axes[r,col].set_yticks([])
        fig.colorbar(heat,ax=axes[:,3],label='1−SSIM（全図0〜2）',shrink=.8)
        fig.suptitle(f'{row.relative_path} / 正解：{row.category_ja} / {scenario} / seed{seed}',fontsize=13)
        stem=row.relative_path.replace('/','_').replace('.png','')
        fig.savefig(visual_dir/f'{scenario}_seed{seed}_{stem}.png',dpi=150);plt.show();plt.close(fig)


## 完了記録・考察の範囲

- 角度を変えた復元図から、向きが固定角度へ偏る可能性を観察する。最も近い角度を選ぶ仕組みや暗記を断定しない。
- B/Cで未知角度の性能が改善するか、正常画像の向きと細部を保てるかを分けて見る。
- 回転拡張だけで復元品質が保証されるわけではなく、構造・容量・損失などの課題も残り得る。
- 向きの変化を撮影方向の変化と仮定し、元ラベルを維持。配置異常のmisplacedは特に種類別の結果を確認する。
- 反射補完・切り取り・補間・縮小が含まれる合成回転。実撮影の変化とは異なる。
- 単一学習シードで、5回のばらつきは角度割り当てによるもの。独立した500枚でも信頼区間でもない。
- 既存テストを繰り返し見た探索的検証。bottleのAUROC改善や実環境への有効性は別途確認する。


In [ ]:
assert all(sha256(ROOT/path)==value for path,value in source_hashes.items()),'既存の読み込み元が変わりました。'
assert all(sha256(ROOT/r['repo_path'])==r['file_sha256'] for r in image_records)
assert all(sha256(ROOT/r['path'])==r['file_sha256'] for r in train_manifest)
assert all(sha256(TRAINING_DIR/k/'model_final.pth')==summaries[k]['model_sha256'] for k in TRAIN_CONDITIONS)
evaluation_config.update(status='completed',completed_at_jst=datetime.now(ZoneInfo('Asia/Tokyo')).isoformat(),
    model_score_count=len(scores_table),test_source_count=100,case_count=len(test_cases))
write_json(EVALUATION_DIR/'config.json',evaluation_config)
session_config.update(status='completed',preprocessing_check_passed=True,
    training_dir=TRAINING_DIR.relative_to(ROOT).as_posix(),source_hashes=source_hashes,
    train_manifest_sha256=object_hash(train_manifest),completed_at_jst=datetime.now(ZoneInfo('Asia/Tokyo')).isoformat())
write_json(SESSION_DIR/'session_config.json',session_config)
files=sorted(p for p in SESSION_DIR.rglob('*') if p.is_file() and p.name!='output_file_hashes.json')
write_json(SESSION_DIR/'output_file_hashes.json',{p.relative_to(SESSION_DIR).as_posix():sha256(p) for p in files})
print('保存先：',SESSION_DIR)
print('学習モデル：',TRAINING_DIR)
print('①前処理照合、②角度別復元、③新規3モデルの比較を保存しました。bottleは対象外です。')


## 参照と確認状態

- [未知回転角度の評価ノートブック](261117transistorの未知回転角度に対する異常検知性能の比較.ipynb)
- [更新回数を揃えた過去の比較](261106transistorの更新回数を揃えた通常学習と回転拡張のAUROC比較.ipynb)
- [memory.md](../../../memory/memory.md)

2026-11-18：コード作成・静的確認のみ。ノートブック・学習・推論・描画・構文テスト・動作テストはアシスタント未実行。
既存の画像・モデル・結果は変更していません。実行時の数値・画像はユーザーが確認してください。
